# Drift Corrector Isolation Test

Exercises the TRIAD-based orientation correction in isolation:

1. **SO(3) log / exp maps** — verify `rotation_log` and `rotation_exp` are mutual inverses (inverse Rodrigues), including the θ→0 and θ→π edge cases.
2. **Partial-correction convergence** — with a fixed TRIAD target, the scaled correction `R_corr = exp(β·log(R_err))` should drive the gyro estimate toward TRIAD, with the error decaying by roughly a factor `(1−β)` per step.
3. **Noisy-TRIAD stress test** — feed a noisy absolute (TRIAD) estimate each step; the correction gain alone should low-pass the noise so the fused estimate tracks the true orientation smoothly, without any pre-filtering of the sensor inputs.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))

import numpy as np
import matplotlib.pyplot as plt

from drone_control_system.ahrs.math_utils import (
    rotation_log, rotation_exp, rodrigues_rotation,
)
from drone_control_system.ahrs.drift_corrector import (
    compute_orientation_error, compute_correction, apply_drift_correction,
)
from drone_control_system.ahrs.triad import triad_init
from drone_control_system.ahrs.euler import rotation_matrix_to_euler

rng = np.random.default_rng(42)

## Scenario 1 — log / exp roundtrip (inverse Rodrigues)

For a batch of random rotation vectors φ (with ‖φ‖ < π), `rotation_log(rotation_exp(φ))` should return φ to machine precision. We also check the documented edge cases: θ→0 returns the zero vector (bypassing the 1/sinθ singularity), and θ→π still recovers the correct angle.

In [ ]:
N = 2000
errs = np.empty(N)
for i in range(N):
    axis = rng.normal(0, 1, 3)
    axis /= np.linalg.norm(axis)
    theta = rng.uniform(0, np.pi - 1e-3)   # keep within (0, pi)
    phi = theta * axis
    phi_rt = rotation_log(rotation_exp(phi))
    errs[i] = np.linalg.norm(phi_rt - phi)

print(f'Roundtrip error  max={errs.max():.2e}  mean={errs.mean():.2e}')
print(f'log(I)                 = {rotation_log(np.eye(3))}')
print(f'log(exp(1e-12 axis))   = {rotation_log(rodrigues_rotation([1,0,0], 1e-12))}')
R_pi = rodrigues_rotation([0, 0, 1], np.pi - 1e-6)
print(f'angle near pi          = {np.linalg.norm(rotation_log(R_pi)):.6f}  (expected ~{np.pi:.6f})')

fig, ax = plt.subplots(figsize=(9, 3))
ax.semilogy(errs, '.', ms=2, alpha=0.4)
ax.set_xlabel('sample'); ax.set_ylabel('roundtrip error')
ax.set_title('log/exp roundtrip error (should be ~1e-15)')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Scenario 2 — Partial-correction convergence

A fixed absolute target `R_triad`; the gyro estimate starts at identity and receives **no** rotation rate, so each step only applies the partial correction
`R_gyro ← exp(β·log(R_triad·R_gyroᵀ))·R_gyro`.
The geodesic error angle ‖log(R_err)‖ should decay geometrically. For a pure SO(3) correction the per-step factor is exactly `(1−β)`, which we overlay as a reference.

In [ ]:
beta = 0.05
n_steps = 200

# Fixed absolute target: 30 deg about a tilted axis
axis = np.array([0.4, -0.7, 0.6]); axis /= np.linalg.norm(axis)
R_triad = rodrigues_rotation(axis, np.deg2rad(30.0))

R_gyro = np.eye(3)
err_angle = np.empty(n_steps)
for k in range(n_steps):
    R_err = compute_orientation_error(R_triad, R_gyro)
    err_angle[k] = np.linalg.norm(rotation_log(R_err))
    R_corr = compute_correction(R_err, beta)
    R_gyro = apply_drift_correction(R_corr, R_gyro)

ref = err_angle[0] * (1 - beta) ** np.arange(n_steps)

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogy(np.rad2deg(err_angle), color='crimson', lw=2, label='error angle')
ax.semilogy(np.rad2deg(ref), 'k--', lw=1, label=f'(1-β)^k reference, β={beta}')
ax.set_xlabel('step'); ax.set_ylabel('error angle (deg)')
ax.set_title('Partial-correction convergence toward fixed TRIAD target')
ax.legend(); ax.grid(True, alpha=0.3, which='both')
plt.tight_layout(); plt.show()

print(f'Initial error : {np.rad2deg(err_angle[0]):.3f} deg')
print(f'Final error   : {np.rad2deg(err_angle[-1]):.4f} deg')
print(f'Max deviation from (1-β)^k reference : {np.abs(err_angle - ref).max():.2e} rad')

## Scenario 3 — Noisy-TRIAD stress test

The drone is static and level (true orientation = identity). Each step builds a TRIAD estimate from **noisy, unfiltered** accel/mag readings, so `R_triad` jitters. With gyro rate = 0, the only smoothing is the correction gain β itself. We compare the raw TRIAD roll/pitch/yaw against the fused estimate, which should be markedly smoother while staying near zero — demonstrating that β replaces the old EMA pre-filtering.

In [ ]:
n_steps = 1500
beta = 0.02
accel_noise = 0.08
mag_noise = 0.04

g_true = np.array([0.0, 0.0, 9.81])   # specific force, level drone (ENU +Z up)
m_true = np.array([0.0, 1.0, 0.0])    # magnetic north

R_gyro = np.eye(3)
triad_eul = np.empty((n_steps, 3))
fused_eul = np.empty((n_steps, 3))

for k in range(n_steps):
    accel = g_true + rng.normal(0, accel_noise, 3)
    mag   = m_true + rng.normal(0, mag_noise, 3)
    R_triad = triad_init(accel, mag)      # normalizes internally

    R_err  = compute_orientation_error(R_triad, R_gyro)
    R_corr = compute_correction(R_err, beta)
    R_gyro = apply_drift_correction(R_corr, R_gyro)

    triad_eul[k] = rotation_matrix_to_euler(R_triad)
    fused_eul[k] = rotation_matrix_to_euler(R_gyro)

labels = ['Roll', 'Pitch', 'Yaw']
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
for i, (ax, lbl) in enumerate(zip(axes, labels)):
    ax.plot(np.rad2deg(triad_eul[:, i]), color='steelblue', lw=0.6, alpha=0.5, label='TRIAD (raw)')
    ax.plot(np.rad2deg(fused_eul[:, i]), color='crimson', lw=1.5, label='Fused')
    ax.axhline(0.0, color='k', ls='--', lw=1)
    ax.set_ylabel(f'{lbl} (deg)'); ax.grid(True, alpha=0.3)
axes[0].legend(loc='upper right', fontsize=8)
axes[-1].set_xlabel('sample')
fig.suptitle(f'Noisy TRIAD vs fused estimate  (β={beta})', fontsize=12)
plt.tight_layout(); plt.show()

warm = n_steps // 3
for i, lbl in enumerate(labels):
    print(f'{lbl:<6}  TRIAD std={np.rad2deg(triad_eul[warm:, i]).std():6.3f} deg   '
          f'fused std={np.rad2deg(fused_eul[warm:, i]).std():6.3f} deg')